## Flow :
#### (Message bits) -> Encoder -> (Codeword) -> Modulation -> AWGN -> Receive Filter -> Decoder -> Estimated message

# 1. Encoder
### QC-LDPC (N, K) 
#### Implements: Base matrix -> Lift to H -> Systematic form -> Generator matrix G -> Encoding

In [1]:
import numpy as np

#### STEP 1: Base Matrix (B) -- dimensions and contents derived from N, K, z

#### n_p = N / z   (number of columns in B)
#### m_p = (N-K)/z (number of rows in B)
#### Entry = cyclic shift amount in [0, z-1], or -1 for an all-zero block.

In [14]:
def gf2_rank(matrix):
    """Rank of a binary matrix over GF(2), via Gaussian elimination."""
    M = matrix.copy() % 2
    rows, cols = M.shape
    rank = 0
    for col in range(cols):
        pivot = None
        for r in range(rank, rows):
            if M[r, col] == 1:
                pivot = r
                break
        if pivot is None:
            continue
        M[[rank, pivot]] = M[[pivot, rank]]
        for r in range(rows):
            if r != rank and M[r, col] == 1:
                M[r] = (M[r] + M[rank]) % 2
        rank += 1
        if rank == rows:
            break
    return rank
 
 
def generate_base_matrix(N, K, z, max_tries=5000, seed=None):
    """
    Automatically size and populate a base matrix B for the requested
    (N, K, z), searching random shift patterns until the lifted H is
    full rank (i.e. it will successfully reduce to systematic form).
 
    Raises ValueError if N, K, z aren't compatible, or RuntimeError if
    no full-rank B is found within max_tries attempts (try a different z).
    """
    if N % z != 0 or (N - K) % z != 0:
        raise ValueError(
            f"z={z} must evenly divide both N={N} and (N-K)={N-K}."
        )
    n_p = N // z
    m_p = (N - K) // z
    if m_p <= 0 or n_p <= m_p:
        raise ValueError(
            f"Invalid dimensions: need 0 < m_p < n_p (got m_p={m_p}, n_p={n_p}). "
            f"Check that 0 < K < N."
        )
 
    rng = np.random.default_rng(seed)
    for _ in range(max_tries):
        B = rng.integers(low=0, high=z, size=(m_p, n_p))
        H = build_parity_check_matrix(B, z)
        if gf2_rank(H) == m_p * z:
            return B
 
    raise RuntimeError(
        f"Could not find a full-rank base matrix for N={N}, K={K}, z={z} "
        f"after {max_tries} tries. Try a different z."
    )

#### STEP 2: Lift B into the full parity-check matrix H

In [2]:
def circulant_permutation_matrix(z, shift):
    """z x z identity matrix, cyclically shifted right by `shift` columns."""
    I = np.eye(z, dtype=int)
    return np.roll(I, shift, axis=1)
 
 
def build_parity_check_matrix(B, z):
    m_p, n_p = B.shape
    H = np.zeros((m_p * z, n_p * z), dtype=int)
    for i in range(m_p):
        for j in range(n_p):
            shift = B[i, j]
            if shift >= 0:                       # -1 would mean "all-zero block"
                block = circulant_permutation_matrix(z, shift)
            else:
                block = np.zeros((z, z), dtype=int)
            H[i*z:(i+1)*z, j*z:(j+1)*z] = block
    return H

#### STEP 3: Convert H to systematic form  H = [P | I]  over GF(2)

In [3]:
def to_systematic_form(H):
    """
    Gaussian elimination over GF(2) to convert H (m x n) into [P | I_m].
    Returns (H_sys, col_order) where col_order records how columns were
    permuted (needed so we know which output columns are the parity bits).
    """
    H_sys = H.copy() % 2
    m, n = H_sys.shape
    col_order = list(range(n))          # track column identity through swaps
 
    # We want the identity to end up in the LAST m columns.
    target_cols = list(range(n - m, n))
 
    for i in range(m):
        target_col = target_cols[i]
 
        # 1. Find a pivot (row with a 1) in the target column, at or below row i
        pivot_row = None
        for r in range(i, m):
            if H_sys[r, target_col] == 1:
                pivot_row = r
                break
 
        if pivot_row is None:
            # No pivot in this column -> swap this column with some earlier
            # column (outside the identity block) that has a usable 1
            swap_col = None
            for c in range(n - m):
                if any(H_sys[r, c] == 1 for r in range(i, m)):
                    swap_col = c
                    break
            if swap_col is None:
                raise ValueError("Matrix is not full rank; cannot form systematic matrix.")
            H_sys[:, [target_col, swap_col]] = H_sys[:, [swap_col, target_col]]
            col_order[target_col], col_order[swap_col] = col_order[swap_col], col_order[target_col]
            # re-find pivot row now that column has changed
            for r in range(i, m):
                if H_sys[r, target_col] == 1:
                    pivot_row = r
                    break
 
        # 2. Move pivot row into position i
        if pivot_row != i:
            H_sys[[i, pivot_row], :] = H_sys[[pivot_row, i], :]
 
        # 3. Eliminate every other 1 in this column (mod 2)
        for r in range(m):
            if r != i and H_sys[r, target_col] == 1:
                H_sys[r, :] = (H_sys[r, :] + H_sys[i, :]) % 2
 
    return H_sys, col_order

#### STEP 4: Derive Generator Matrix G = [I<sub>k</sub> | P<sup>T</sup>]

In [4]:
def derive_generator_matrix(H_sys, k):
    n = H_sys.shape[1]
    P = H_sys[:, :k]              # H_sys = [P | I_m]
    G = np.hstack((np.eye(k, dtype=int), P.T)) % 2
    return G

#### STEP 5: Encode  v = u . G (mod 2)

In [5]:
def encode(u, G):
    u = np.array(u, dtype=int)
    v = (u @ G) % 2
    return v

#### Verification: check every possible K-bit message gives a valid codeword

In [6]:
def verify_all_codewords(H_sys, G, K, verbose=True):
    """
    Encodes every possible K-bit message and checks that H_sys . v^T = 0
    for each resulting codeword v. Works for any K (not hardcoded to 3).
 
    Returns True if every codeword is valid, False otherwise.
    """
    num_messages = 2 ** K
    all_ok = True
 
    if verbose:
        print(f"Verifying all {num_messages} possible {K}-bit messages "
              f"give valid codewords (H.v^T = 0):")
 
    for i in range(num_messages):
        u_test = [(i >> b) & 1 for b in range(K - 1, -1, -1)]
        v_test = encode(u_test, G)
        syndrome = (H_sys @ v_test) % 2
        ok = not syndrome.any()
        all_ok &= ok
        if verbose:
            print(f"  u={u_test} -> v={v_test}  syndrome={syndrome}  {'OK' if ok else 'FAIL'}")
 
    if verbose:
        print("\nAll codewords valid!" if all_ok else "\nSOMETHING IS WRONG")
 
    return all_ok

# 2. BPSK Modulation and AWGN

#### STEP 6: BPSK Modulation

In [7]:
def bpsk_modulate(v):
    """
    Map bits to BPSK symbols: 0 -> +1, 1 -> -1
    v : array of 0/1 codeword bits
    returns : array of +1/-1 floats
    """
    v = np.array(v, dtype=int)
    x = 1 - 2 * v          # 0 -> 1-0=1,  1 -> 1-2=-1
    return x.astype(float)

#### STEP 7: AWGN Channel

In [8]:
def awgn_noise_std(ebno_db, rate):
    """
    Convert Eb/N0 (in dB) into the noise standard deviation for BPSK
    symbols of unit energy (Es = 1), taking the code rate into account.
 
    Eb = Es / rate          (less-efficient/lower-rate codes spend more
                              energy per information bit)
    N0 = Eb / (Eb/N0)_linear
    sigma^2 = N0 / 2         (noise variance per real dimension)
    """
    ebno_linear = 10 ** (ebno_db / 10)
    Es = 1.0
    Eb = Es / rate
    N0 = Eb / ebno_linear
    sigma = np.sqrt(N0 / 2)
    return sigma

In [9]:
def awgn_channel(x, sigma, seed=None):
    """
    Pass BPSK symbols x through an AWGN channel.
    Adds i.i.d. Gaussian noise ~ N(0, sigma^2) to each symbol.
    Returns continuous, real-valued received samples.
    """
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=sigma, size=np.shape(x))
    r = x + noise
    return r

### Full pipeline

In [24]:
N = 6
K = 4
z = 2        # must evenly divide both N and (N-K)

In [25]:
B = generate_base_matrix(N, K, z, seed=0)
print(f"N={N}, K={K}, z={z}  ->  base matrix B (shape {B.shape}):")
print(B, "\n")

N=6, K=4, z=2  ->  base matrix B (shape (1, 3)):
[[1 1 1]] 



In [26]:
H = build_parity_check_matrix(B, z)
print("Parity-check matrix H (lifted from base matrix):")
print(H, "\n")

Parity-check matrix H (lifted from base matrix):
[[0 1 0 1 0 1]
 [1 0 1 0 1 0]] 



In [28]:
H_sys, col_order = to_systematic_form(H)
print("Systematic parity-check matrix H_sys = [P | I]:")
print(H_sys, "\n")

Systematic parity-check matrix H_sys = [P | I]:
[[1 0 1 0 1 0]
 [0 1 0 1 0 1]] 



In [29]:
G = derive_generator_matrix(H_sys, K)
print("Generator matrix G = [I | P^T]:")
print(G, "\n")

Generator matrix G = [I | P^T]:
[[1 0 0 0 1 0]
 [0 1 0 0 0 1]
 [0 0 1 0 1 0]
 [0 0 0 1 0 1]] 



In [30]:
# Encode an example K-bit message (random, since K is configurable)
rng_demo = np.random.default_rng(1)
u = rng_demo.integers(0, 2, size=K).tolist()
v = encode(u, G)
print(f"Message u = {u}")
print(f"Codeword v = {v}\n")

Message u = [0, 1, 1, 1]
Codeword v = [0 1 1 1 1 0]



In [31]:
# --- Modulate + transmit through AWGN ---
R = K / N                       # code rate = 0.5
ebno_db = 3.0                   # try e.g. 0, 3, 6, 10 dB

x = bpsk_modulate(v)
sigma = awgn_noise_std(ebno_db, R)
r = awgn_channel(x, sigma)

print(f"BPSK symbols x        = {x}")
print(f"Noise std (sigma)     = {sigma:.4f}  (Eb/N0 = {ebno_db} dB, rate = {R})")
print(f"Received samples r    = {np.round(r, 4)}\n")

BPSK symbols x        = [ 1. -1. -1. -1. -1.  1.]
Noise std (sigma)     = 0.6131  (Eb/N0 = 3.0 dB, rate = 0.6666666666666666)
Received samples r    = [ 1.0786 -0.7943 -2.4095 -0.8129  0.1882  0.8334]



In [32]:
# A quick hard-decision sanity check: r>0 -> bit 0, r<0 -> bit 1
hard_bits = (r < 0).astype(int)
print(f"Hard-decision bits    = {hard_bits}")
print(f"(compare to original codeword v = {v})")

Hard-decision bits    = [0 1 1 1 0 0]
(compare to original codeword v = [0 1 1 1 1 0])


In [33]:
# --- Sanity check: H_sys . v^T should be all zeros for EVERY message ---
verify_all_codewords(H_sys, G, K)

Verifying all 16 possible 4-bit messages give valid codewords (H.v^T = 0):
  u=[0, 0, 0, 0] -> v=[0 0 0 0 0 0]  syndrome=[0 0]  OK
  u=[0, 0, 0, 1] -> v=[0 0 0 1 0 1]  syndrome=[0 0]  OK
  u=[0, 0, 1, 0] -> v=[0 0 1 0 1 0]  syndrome=[0 0]  OK
  u=[0, 0, 1, 1] -> v=[0 0 1 1 1 1]  syndrome=[0 0]  OK
  u=[0, 1, 0, 0] -> v=[0 1 0 0 0 1]  syndrome=[0 0]  OK
  u=[0, 1, 0, 1] -> v=[0 1 0 1 0 0]  syndrome=[0 0]  OK
  u=[0, 1, 1, 0] -> v=[0 1 1 0 1 1]  syndrome=[0 0]  OK
  u=[0, 1, 1, 1] -> v=[0 1 1 1 1 0]  syndrome=[0 0]  OK
  u=[1, 0, 0, 0] -> v=[1 0 0 0 1 0]  syndrome=[0 0]  OK
  u=[1, 0, 0, 1] -> v=[1 0 0 1 1 1]  syndrome=[0 0]  OK
  u=[1, 0, 1, 0] -> v=[1 0 1 0 0 0]  syndrome=[0 0]  OK
  u=[1, 0, 1, 1] -> v=[1 0 1 1 0 1]  syndrome=[0 0]  OK
  u=[1, 1, 0, 0] -> v=[1 1 0 0 1 1]  syndrome=[0 0]  OK
  u=[1, 1, 0, 1] -> v=[1 1 0 1 1 0]  syndrome=[0 0]  OK
  u=[1, 1, 1, 0] -> v=[1 1 1 0 0 1]  syndrome=[0 0]  OK
  u=[1, 1, 1, 1] -> v=[1 1 1 1 0 0]  syndrome=[0 0]  OK

All codewords valid!


True